<div style="padding: 20px 24px; border-radius: 12px; background: linear-gradient(110deg, #12304a, #187b8f); color: white; margin-bottom: 16px;">
  <div style="font-size: 0.78em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.85;">HydroMT-SFINCS | Example 05A</div>
  <h1 style="color: white; margin: 8px 0 4px;">SFINCS results: plot maximum water depth</h1>
  <div style="opacity: 0.95;">Visualize maximum water depths directly from regular-grid or quadtree SFINCS output.</div>
</div>


This notebook plots the maximum water depth (`hmax`) stored in `sfincs_map.nc`. It applies to models **without subgrid tables**, including regular-grid and quadtree models.

For models with subgrid tables, use maximum water levels and a high-resolution DEM instead. Follow [Example 05B: Downscale maximum water depth](5b_plot_downscaled_results_hmax.ipynb).


# Determine maximum water depth directly from SFINCS output

### Steps

1. Read model results and inspect the model layout.
2. Optionally write maximum water depth to a GeoTIFF (regular grids only).
3. Optionally mask permanent water and apply a minimum depth threshold.
4. Plot maximum water depth.


For models without subgrid tables, SFINCS computes water depth from water levels and bed elevations and stores maximum water depth (`hmax`) in `sfincs_map.nc`. HydroMT-SFINCS reads both regular-grid and quadtree results, and `plot_basemap` can plot the resulting maximum-depth field.

The optional GSWO mask below is a regular-grid raster calculation. For quadtree results, skip that step and plot `hmax` directly or use a vector-based mask.


In [ ]:
from os.path import join
import matplotlib.pyplot as plt

from hydromt_sfincs import SfincsModel

### Step 1: Read model results

The map output uses staggered-grid conventions for regular grids and an unstructured mesh for quadtree grids. `mod.output.read()` handles these formats and makes the output variables available through `mod.output.data`.


In [ ]:
sfincs_root = "sfincs_compound"  # (relative) path to sfincs root
mod = SfincsModel(sfincs_root, data_libs=["artifact_data"], mode="r")

In [ ]:
# we can simply read the model results (sfincs_map.nc and sfincs_his.nc) using the output.read method
mod.output.read()

# the following variables have been found
list(mod.output.data.keys())

Inspect the model layout. Boundary-cell plotting is disabled because it is not supported for quadtree grids.


In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None,
    bmap="sat",
    figsize=(11, 7),
    zoomlevel=12,
    plot_bounds=False,
)

### Step 2: Optionally write maximum water depth to GeoTIFF (regular grids)


<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#128161;</span> NOTE</strong> This optional raster export is for regular-grid DataArrays. Quadtree output is unstructured; plot it directly or rasterize it with an appropriate workflow before writing a GeoTIFF.</div>


In [ ]:
# Optional regular-grid GeoTIFF export
# from hydromt_sfincs import writers
# hmax = mod.output.data["hmax"].max(dim="timemax")
# hmax.attrs.update(long_name="maximum water depth", unit="m")
# writers.write_raster(
#     hmax,
#     join(mod.root.path, "gis"),
#     compress="LZW",
# )

### Step 3: Optionally mask permanent water and apply a minimum depth threshold


A Global Surface Water Occurrence (GSWO) raster can be used to mask permanent water on a regular grid. This reprojection step is raster-specific and should be skipped/replaced for quadtree output.


In [ ]:
# Maximum water depth from the model output
da_hmax = mod.output.data["hmax"].max(dim="timemax")

if mod.grid_type == "regular":
    # Permanent water is where water occurrence is greater than 5%.
    gswo = mod.data_catalog.get_rasterdataset("gswo", geom=mod.region, buffer=10)
    gswo_mask = gswo.raster.reproject_like(mod.grid.data, method="max") <= 5
    da_hmax = da_hmax.where(gswo_mask)
else:
    print("Skipping GSWO raster masking for quadtree output; plotting hmax directly.")

da_hmax.attrs.update(long_name="flood depth", unit="m")

<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#9888;&#65039;</span> NOTE</strong> To read GSWO data for another region, add the source to your data catalog (for example, with <code>mod.data_catalog.from_yml('/path/to/data_catalog.yml')</code>). This raster-mask step applies only to regular grids.</div>


### Step 4: Plot maximum water depth

Plot the maximum water depth, with model geometries such as discharge source points and observation gauges where available.


In [ ]:
# minimum flood depth [m] to plot
hmin = 0.05
da_hmax = da_hmax.where(da_hmax > hmin)

In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None,
    figsize=(8, 6),
    variable=da_hmax,
    plot_bounds=False,
    plot_geoms=False,
    bmap="sat",
    zoomlevel=12,
    vmin=0,
    vmax=2.0,
    cmap=plt.cm.viridis,
    cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
)
ax.set_title(f"SFINCS maximum water depth")
# plt.savefig(join(mod.root, 'figs', 'hmax.png'), dpi=225, bbox_inches="tight")